
# Stress-testing NeuroSAT — 30-minute worksheet

The aim is to distinguish two different kinds of failure:

$$
\text{failure on unfamiliar data}
\qquad\text{versus}\qquad
\text{a limitation of the architecture itself}.
$$

A suggested pace is:

- **5 minutes:** setup, one $SR(n)$ example, and the model interface;
- **12 minutes:** vary a supplied formula generator and search for errors;
- **8 minutes:** read through a SAT/UNSAT pair that NeuroSAT cannot distinguish;
- **5 minutes:** open-ended discussion.

The neural network in this notebook is a small classroom reimplementation of
the NeuroSAT message-passing architecture. It is **not** the published NeuroSAT
checkpoint, so do not compare its numerical accuracy with the paper.

The support code in `neurosat_lab.py` was written for this workshop. The
message-passing architecture and the $SR(n)$ construction follow Selsam et al.,
*Learning a SAT Solver from Single-Bit Supervision* (ICLR 2019).



## 0. Setup

A CNF formula is represented by a list of clauses. Positive integers represent
positive literals and negative integers represent negated literals. For example,

```python
[(1, -2), (2, 3)]
```

represents

$$
(x_1\lor \neg x_2)\land(x_2\lor x_3).
$$

The exact solver in `neurosat_lab.py` is a small DPLL implementation used only
to supply ground-truth SAT/UNSAT labels for the exercise.


In [ ]:

from pathlib import Path
import random

import numpy as np
import pandas as pd

import neurosat_lab as ns

SEED = 4
rng = random.Random(SEED)
np.random.seed(SEED)

# Keep the notebook, support module, and checkpoint in the same directory.
CHECKPOINT = Path.cwd() / "toy_neurosat_checkpoint.npz"
if not CHECKPOINT.exists():
    raise FileNotFoundError(
        "Could not find toy_neurosat_checkpoint.npz. "
        "Place it in the same directory as this notebook."
    )

model, checkpoint_info = ns.load_checkpoint(str(CHECKPOINT))
pd.Series(checkpoint_info, name="checkpoint metadata")



## 1. What is $SR(n)$?

Fix variables

$$
x_1,\ldots,x_n.
$$

To generate **one random clause**:

1. draw
   $$
   B\sim \operatorname{Bernoulli}(0.7),
   \qquad
   G\sim \operatorname{Geometric}(0.4),
   $$
   where $G\in\{1,2,\ldots\}$;
2. set the proposed width
   $$
   K=1+B+G;
   $$
3. use the actual width
   $$
   W=\min\{K,n\};
   $$
4. choose $W$ distinct variables uniformly without replacement;
5. independently negate each chosen variable with probability $1/2$.

Fresh randomness is used for every new clause.

Now keep adding random clauses until the formula becomes unsatisfiable for the
first time:

$$
F^- = C_1\land\cdots\land C_{m-1}\land C_m
$$

is UNSAT, but $C_1\land\cdots\land C_{m-1}$ is SAT. Flip the sign of one
literal occurrence in the final clause $C_m$, obtaining $C'_m$, and set

$$
F^+ = C_1\land\cdots\land C_{m-1}\land C'_m.
$$

Then $F^+$ is satisfiable. Thus one draw from $SR(n)$ is the matched pair

$$
(F^+,F^-),
$$

which has opposite labels but differs in only one literal occurrence.


In [ ]:

# Generate one small matched pair and compare exact labels with neural predictions.
example_sat, example_unsat = ns.generate_sr_pair(
    n_vars=7,
    rng=random.Random(17),
)

rows = []
for formula in (example_sat, example_unsat):
    exact_label, witness = ns.solve_exact(formula)
    probability = float(ns.predict_probabilities(model, [formula])[0])
    decoded = ns.decode_assignment(model, formula)

    rows.append({
        "formula": formula.name,
        "exact label": "SAT" if exact_label else "UNSAT",
        "P(SAT)": round(probability, 4),
        "decoded witness verified":
            decoded is not None and ns.satisfies(formula, decoded),
    })

pd.DataFrame(rows)



### Viewing a generated formula

The code stores a clause such as $(x_1\lor\neg x_3)$ as the tuple `(1, -3)`.
This signed-integer representation is convenient for computation but not for
reading.

Use

```python
ns.display_cnf(formula)
```

to render a formula in standard logical notation. By default it displays at
most $12$ clauses. Use `max_clauses=None` to display the complete formula, or
reduce `max_clauses` when the generated formula is large.


In [ ]:

# Display the satisfiable member of the example pair in ordinary notation.
ns.display_cnf(
    example_sat,
    max_clauses=8,
    clauses_per_line=2,
)



### Quick check

- Which entries in the table are mathematical facts, and which are predictions?
- If the network reports $P(\mathrm{SAT})=0.99$, does that by itself prove that
  the formula is satisfiable?
- If a decoded assignment satisfies every clause, what has been proved?



## 2. Challenge: make the model fail

The classroom checkpoint was trained on small $SR(n)$ formulas with roughly
$5\leq n\leq 10$. The next cell contains the complete experiment. You only
need to change the parameters at the top and rerun it.

Two formula generators are provided:

1. **`"sr"`:** matched $SR(n)$ pairs, as defined above;
2. **`"random_kcnf"`:** random formulas in which every clause contains exactly
   $k$ distinct variables, independently negated with probability $1/2$.

For random $k$-CNF, the parameter `CLAUSE_RATIO` means

$$
\frac{\text{number of clauses}}{\text{number of variables}}.
$$

Suggested starting points are:

```python
# Larger formulas from the same generator
FAMILY = "sr"
N_VARS = 16

# Random 2-CNF
FAMILY = "random_kcnf"
N_VARS = 24
CLAUSE_WIDTH = 2
CLAUSE_RATIO = 1.0

# Random 3-CNF
FAMILY = "random_kcnf"
N_VARS = 20
CLAUSE_WIDTH = 3
CLAUSE_RATIO = 4.3
```

The exact DPLL solver supplies the ground-truth label. The neural network
supplies only a prediction. The experiment reports the formulas on which the
network is most confidently wrong.


In [ ]:

# ============================================================
# PARAMETERS TO CHANGE
# ============================================================

FAMILY = "sr"              # Choose "sr" or "random_kcnf".
N_VARS = 16                # Number of Boolean variables.
QUERY_BUDGET = 40          # Maximum number of formulas tested.
EXPERIMENT_SEED = 29

# These two parameters are used only when FAMILY == "random_kcnf".
CLAUSE_WIDTH = 3
CLAUSE_RATIO = 4.3         # Number of clauses is about CLAUSE_RATIO * N_VARS.

# ============================================================
# FORMULA GENERATION
# You should not need to change the code below.
# ============================================================

experiment_rng = random.Random(EXPERIMENT_SEED)


def generate_formula_batch():
    """Generate at most QUERY_BUDGET formulas using the chosen family."""
    formulas = []

    if FAMILY == "sr":
        # One call produces a matched SAT/UNSAT pair.
        while len(formulas) < QUERY_BUDGET:
            sat_formula, unsat_formula = ns.generate_sr_pair(
                n_vars=N_VARS,
                rng=experiment_rng,
            )
            formulas.extend([sat_formula, unsat_formula])

    elif FAMILY == "random_kcnf":
        n_clauses = max(1, round(CLAUSE_RATIO * N_VARS))

        for index in range(QUERY_BUDGET):
            formulas.append(
                ns.random_kcnf(
                    n_vars=N_VARS,
                    n_clauses=n_clauses,
                    k=CLAUSE_WIDTH,
                    rng=experiment_rng,
                    name=f"random-{CLAUSE_WIDTH}-CNF-{index}",
                )
            )

    else:
        raise ValueError(
            'FAMILY must be either "sr" or "random_kcnf".'
        )

    return formulas[:QUERY_BUDGET]


# ============================================================
# EVALUATION
# The exact solver supplies the label; NeuroSAT supplies P(SAT).
# ============================================================

formulas = generate_formula_batch()
records = []

for index, formula in enumerate(formulas):
    exact_label, _ = ns.solve_exact(formula)
    p_sat = float(ns.predict_probabilities(model, [formula])[0])
    predicted_label = p_sat >= 0.5

    records.append({
        "index": index,
        "exact label": "SAT" if exact_label else "UNSAT",
        "predicted label": "SAT" if predicted_label else "UNSAT",
        "P(SAT)": p_sat,
        "error": predicted_label != exact_label,
        "wrong-answer confidence":
            ns.wrong_answer_confidence(p_sat, int(exact_label)),
    })

results = pd.DataFrame(records)

print("Experiment:")
print("  family:", FAMILY)
print("  variables:", N_VARS)
print("  formulas tested:", len(results))
print("  errors:", int(results["error"].sum()))
print("  accuracy:", round(1 - results["error"].mean(), 3))

errors = (
    results[results["error"]]
    .sort_values("wrong-answer confidence", ascending=False)
)

if errors.empty:
    print("\nNo misclassification was found in this sample.")
    print("Change the parameters and run the cell again.")
else:
    print("\nMost confident errors:")
    display(
        errors[
            [
                "index",
                "exact label",
                "predicted label",
                "P(SAT)",
                "wrong-answer confidence",
            ]
        ].head(5)
    )

    best_index = int(errors.iloc[0]["index"])
    best_formula = formulas[best_index]

    print("\nMost confidently misclassified formula:")
    ns.display_cnf(
        best_formula,
        max_clauses=10,
        clauses_per_line=2,
    )



### Inspect any formula from the experiment

The `formulas` list contains every generated formula, and the `index` column in
the results table gives its position in that list. Change `FORMULA_INDEX` below
to inspect any formula in standard notation.


In [ ]:

# Choose any integer from 0 to len(formulas) - 1.
# If an error was found, this starts with the most confident one.
FORMULA_INDEX = (
    int(errors.iloc[0]["index"])
    if not errors.empty
    else 0
)

formula_to_view = formulas[FORMULA_INDEX]
row = results.loc[results["index"] == FORMULA_INDEX].iloc[0]

print("index:", FORMULA_INDEX)
print("exact label:", row["exact label"])
print("predicted label:", row["predicted label"])
print("P(SAT):", round(float(row["P(SAT)"]), 4))

ns.display_cnf(
    formula_to_view,
    max_clauses=12,
    clauses_per_line=2,
)



### Record what happened

Change the parameters and rerun the experiment at least once. Record:

1. which generator and parameters you used;
2. how many formulas were tested;
3. the most confident error you found;
4. whether the failure looks like ordinary distribution shift.

A failed prediction on unfamiliar data does not yet prove an architectural
limitation. The next example does.



## 3. A SAT/UNSAT pair that NeuroSAT cannot distinguish

Consider

$$
\begin{aligned}
F_{\mathrm{SAT}}
={}&
(x_1\vee\neg x_3)
\wedge(\neg x_1\vee x_3)\\
&\wedge(x_1\vee x_2)
\wedge(\neg x_1\vee\neg x_2)\\
&\wedge(x_2\vee x_3)
\wedge(\neg x_2\vee\neg x_3),
\end{aligned}
$$

and

$$
\begin{aligned}
F_{\mathrm{UNSAT}}
={}&
(x_1\vee\neg x_3)
\wedge(\neg x_1\vee x_3)\\
&\wedge(x_1\vee x_2)
\wedge(\neg x_1\vee\neg x_2)\\
&\wedge(x_2\vee\neg x_3)
\wedge(\neg x_2\vee x_3).
\end{aligned}
$$

The formula

$$
(x\vee\neg y)\land(\neg x\vee y)
$$

expresses $x=y$, while

$$
(x\vee y)\land(\neg x\vee\neg y)
$$

expresses $x\neq y$.

Therefore $F_{\mathrm{SAT}}$ says

$$
x_1=x_3,\qquad x_1\neq x_2,\qquad x_2\neq x_3,
$$

and is satisfied by $(x_1,x_2,x_3)=(1,0,1)$.

By contrast, $F_{\mathrm{UNSAT}}$ says

$$
x_1=x_3,\qquad x_1\neq x_2,\qquad x_2=x_3,
$$

which is contradictory.


In [ ]:

# Confirm the two hand calculations and compare the neural predictions.
F_sat, F_unsat = ns.three_variable_indistinguishable_pair()

for formula in (F_sat, F_unsat):
    label, witness = ns.solve_exact(formula)
    p_sat = float(ns.predict_probabilities(model, [formula])[0])

    print(formula.name)
    print("  exact label:", "SAT" if label else "UNSAT")
    print("  witness:", witness)
    print("  neural P(SAT):", repr(p_sat))
    print("  literal occurrence degrees:", ns.literal_occurrence_degrees(formula))
    print()



### Why the predictions must be identical

NeuroSAT begins with no variable names or node identifiers. Every literal node
starts with the same learned vector

$$
\ell_{\square x_i}^{(0)}=L_{\mathrm{init}},
$$

and every clause node starts with the same learned vector

$$
c_j^{(0)}=C_{\mathrm{init}}.
$$

In both formulas:

- every clause contains exactly two literals;
- every literal occurs in exactly two clauses;
- every literal has exactly one complementary literal.

We prove inductively that, at every message-passing round $t$, all literal
embeddings are equal to a common vector $\ell_t$, and all clause embeddings are
equal to a common vector $c_t$.

This is true at $t=0$ by initialization. Suppose it is true at round $t$.

Each clause receives messages from exactly two literals, both carrying
$\ell_t$. Hence every clause receives the same aggregate message and is updated
to the same vector $c_{t+1}$.

Each literal then receives messages from exactly two clauses, both carrying
$c_{t+1}$, together with the embedding $\ell_t$ of its complement. Hence every
literal receives the same inputs and is updated to the same vector
$\ell_{t+1}$.

The induction follows. The final literal votes, and therefore their average,
are identical in the two formulas:

$$
\operatorname{NeuroSAT}(F_{\mathrm{SAT}})
=
\operatorname{NeuroSAT}(F_{\mathrm{UNSAT}}).
$$

One formula is SAT and the other is UNSAT, so at least one prediction must be
wrong. No additional training data and no increase in the number of
message-passing rounds can repair this particular failure.



### Arbitrarily large examples can be generated the same way

Arrange variables $x_1,\ldots,x_r$ around a cycle and impose, for each $i$,

$$
x_i=x_{i+1}
\qquad\text{or}\qquad
x_i\neq x_{i+1},
$$

with indices taken modulo $r$. Encode each equality or inequality by the same
pair of $2$-clauses used above.

Going once around the cycle returns to $x_1$, so the formula is satisfiable
exactly when the number of inequality constraints is even.

Every formula in this family still has:

- clauses of width $2$;
- exactly two clause occurrences for every literal;
- one complement for every literal.

The same induction therefore gives SAT/UNSAT pairs of arbitrarily large size
that NeuroSAT cannot distinguish.



## 4. Open-ended discussion

- What is the difference between an error caused by distribution shift and the
  indistinguishability result above?
- Is the distinguishing information absent from the formula encoding, or is it
  present but inaccessible to this message-passing architecture?
- If a neural model proposes a satisfying assignment and a classical checker
  verifies it, which part of the conclusion is statistical and which part is
  logically certain?
- What additional supervision might change what NeuroSAT learns internally?



## References

- D. Selsam, M. Lamm, B. Bünz, P. Liang, L. de Moura, and D. L. Dill,
  *Learning a SAT Solver from Single-Bit Supervision*, ICLR 2019.
- The explicit regular SAT/UNSAT indistinguishability phenomenon is discussed
  in recent work on the expressive limitations of GNN-based SAT solvers.
